This notebook is for modeling and evaluating technique classification from the SemEval dataset. Given a span predicted to be propaganda, this model will identify which propaganda techniques, if any, were used.

In [91]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from pathlib import Path
from transformers import AutoModel, PreTrainedModel, AutoConfig, AutoTokenizer, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers.utils.notebook import NotebookProgressCallback
import gdown
import zipfile
import os
import shutil

In [104]:
#Download models if not already
def setup_models(file_id, model_folder_name):
    base_dir = Path("..")
    model_dir = base_dir / "models"
    target_path = model_dir / model_folder_name
    zip_temp = model_dir / f"{model_folder_name}.zip"

    if not (target_path / "pytorch_model.bin").exists():
        print(f"Model not found. Downloading {model_folder_name} from Google Drive...")
        model_dir.mkdir(exist_ok=True)

        url = f'https://drive.google.com/uc?id={file_id}'

        try:
            gdown.download(url, str(zip_temp), quiet=False)

            print("Extracting...")
            with zipfile.ZipFile(zip_temp, 'r') as zip_ref:
                internal_zip_folder = zip_ref.namelist()[0].split('/')[0]
                zip_ref.extractall(model_dir)

            extracted_path = model_dir / internal_zip_folder
            if extracted_path != target_path:
                if target_path.exists(): shutil.rmtree(target_path)
                os.rename(extracted_path, target_path)

            os.remove(zip_temp)
            print(f"Setup complete: {target_path}")
            return True
        except Exception as e:
            print(f"Download failed: {e}")
            print("Ensure the File ID is correct and permissions are set to 'Anyone with the link'.")
            return False
    else:
        print(f"Model weights detected locally at {target_path}")
        return True

#Use the ID of the zip file here
model_already_trained = setup_models('13SC1hSucXUmTtdNKRK8TgytdDNJ4yqXi', 'semeval-roberta-classifier')

Model not found. Downloading semeval-roberta-classifier from Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=13SC1hSucXUmTtdNKRK8TgytdDNJ4yqXi
From (redirected): https://drive.google.com/uc?id=13SC1hSucXUmTtdNKRK8TgytdDNJ4yqXi&confirm=t&uuid=87c7bdf0-bfe7-4f8b-a81a-3058d74a4fc6
To: /Users/frankiepike/Library/Mobile Documents/com~apple~CloudDocs/MADS Projects/Claim-Aware_Propaganda_Scanner/models/semeval-roberta-classifier.zip
100%|██████████| 438M/438M [00:09<00:00, 45.7MB/s] 


Extracting...
Setup complete: ../models/semeval-roberta-classifier


In [105]:
#Load technique classification data
df_tc = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
df_tc.head()

,article_id,text_content,span_text,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,0.250000,0,1.000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,111111111,Next plague outbreak in Madagascar could be 's...,"a very, very different",0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,111111111,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,0.483333,0,0.863636,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,111111111,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,0.000000,0,1.000000,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [106]:
#Split the data by article rather than by span to avoid any data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_tc, groups=df_tc['article_id']))
train_df = df_tc.iloc[train_idx].reset_index(drop=True)
test_df = df_tc.iloc[test_idx].reset_index(drop=True)

#Verify the split
print(f"Total spans: {len(df_tc)}, total articles: {df_tc['article_id'].nunique()}")
print(f"Train spans: {len(train_df)} ({train_df['article_id'].nunique()} articles)")
print(f"Test spans:  {len(test_df)} ({test_df['article_id'].nunique()} articles)")

#Check for leakage (should be 0)
overlap = set(train_df['article_id']).intersection(set(test_df['article_id']))
print(f"Number of overlapping articles: {len(overlap)}")

Total spans: 7587, total articles: 357
Train spans: 5856 (285 articles)
Test spans:  1731 (72 articles)
Number of overlapping articles: 0


In [107]:
#Identify feature columns versus input versus output
feature_cols = ['sentiment', 'punct_count', 'lexical_diversity']
label_cols = [c for c in train_df.columns if c not in (['article_id', 'text_content', 'span_text'] + feature_cols)]

In [108]:
#Using a hybrid model to make the most of RoBERTa and our covariates
class RoBERTaHybrid(nn.Module):
    def __init__(self, model_name, num_labels, num_extra_features):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(768 + num_extra_features, num_labels)
        self.loss_fct = nn.BCEWithLogitsLoss() #For Multi-label

    def forward(self, input_ids, attention_mask, extra_features, labels=None):
        #Get text representation from RoBERTa
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)

        #CONCATENATE: Text features + Manual features
        combined_input = torch.cat((pooled_output, extra_features), dim=1)

        logits = self.classifier(combined_input)

        loss = None
        if labels is not None:
            loss = self.loss_fct(logits, labels.float())

        return SequenceClassifierOutput(loss=loss, logits=logits)

In [109]:
#Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = RoBERTaHybrid("roberta-base", num_labels=len(label_cols), num_extra_features=len(feature_cols))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [110]:
#Put input text data in a format Hugging Face knows how to use
def preprocess_function(examples):
    # okenize the text
    result = tokenizer(examples["span_text"], padding="max_length", truncation=True, max_length=128)

    #Bundle the manual features into a list of floats
    extra_feats = []
    for i in range(len(examples["span_text"])):
        extra_feats.append([examples[col][i] for col in feature_cols])
    result["extra_features"] = extra_feats

    #Bundle the technique labels into a list of floats (Multi-label)
    label_matrix = []
    for i in range(len(examples["span_text"])):
        label_matrix.append([examples[col][i] for col in label_cols])
    result["labels"] = label_matrix

    return result

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)

Map:   0%|          | 0/5856 [00:00<?, ? examples/s]

Map:   0%|          | 0/1731 [00:00<?, ? examples/s]

In [111]:
#Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "extra_features", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "extra_features", "labels"])

In [112]:
training_args = TrainingArguments(
    output_dir=final_model_path,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.4687776063061354e-05,
    per_device_train_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.0829185996302097,
    load_best_model_at_end=True
)

In [113]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    #Convert logits to probabilities
    probs = 1 / (1 + np.exp(-logits))
    #Standard threshold of 0.6, which is slightly higher than the usual 0.5, so we want to be careful about identifying propaganda techniques
    predictions = (probs > 0.6).astype(int)

    #Calculate metrics, 'micro' is generally preferred for multi-label classification
    f1 = f1_score(labels, predictions, average='micro')
    precision = precision_score(labels, predictions, average='micro', zero_division=0)
    recall = recall_score(labels, predictions, average='micro', zero_division=0)

    #We can also use fbeta_score with beta=0.75 to favor precision slightly in the future, if desired
    return {
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [114]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [115]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(MODEL_DIR)
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model was already loaded from disk. Skipping training.")
    trainer.remove_callback(NotebookProgressCallback)

Model was already loaded from disk. Skipping training.


In [ ]:
#Evaluate performance on the test dataset
trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\n" + "="*30)
print("FINAL MODEL PERFORMANCE")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"F1 Score:  {test_results['eval_f1_score']:.4f}")
print("="*30)

/Users/frankiepike/ds_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
